# Cloud removal and super-resolution evaluation on Colab T4

This notebook evaluates **Former (ECRformer)** and **CNN (DSen2-CR)** on 50 paired scenes, then optionally runs **OpenSR** on both outputs. It reports per-scene metrics and dataset mean ± standard deviation, and writes CSV and GeoTIFF results.

## Data contract
Each scene folder must contain `s1.tif` (2 bands, VV then VH, Sentinel-1 GRD backscatter in dB), `s2_cloudy.tif` (13 Sentinel-2 bands in the project order, preferably DN 0–10000), `ground_truth.tif` (13 cloud-free Sentinel-2 bands in the same band order/grid), and optionally `valid_mask.tif` (one band; nonzero = valid). Files must share CRS, pixel dimensions, and geotransform. Ground truth should be from a suitably close date and independently selected clear scene.

Use a zip containing `scene_01/...` through `scene_10/...`. The project bundle zip must contain this project's source folders plus `model.ckpt`, `dsen2-cr-main/checkpoints/model_SARcarl.torch.npz`, and `diffffusiosm/` with its OpenSR model weights. Do not include `.venv`, `.venv37`, `.conda37`, or large unrelated dataset folders in the source bundle.

**Important SR metric limit:** if the reference is 10 m, OpenSR predictions are averaged back to the reference grid before scoring. Those scores are 10 m-equivalent agreement, not proof of true 2.5 m detail accuracy. Native-resolution SR accuracy requires a co-registered 2.5 m ground truth.

## 1. Select GPU runtime and install the remaining libraries
In Colab choose **Runtime → Change runtime type → T4 GPU**. Run the check before installing anything; this install cell deliberately does not reinstall PyTorch.

In [ ]:
import torch
assert torch.cuda.is_available(), 'CUDA is unavailable. Select a T4 GPU runtime, then reconnect.'
print(torch.cuda.get_device_name(0))
!pip -q install timm einops scipy scikit-image rasterio pandas
import torch
print('PyTorch:', torch.__version__, '| CUDA:', torch.version.cuda, '| GPU:', torch.cuda.get_device_name(0))

## 2. Mount Drive and unpack your project/data archives
Upload `colab_project_bundle.zip` and `sen12ms_cr_50_pairs.zip` to a folder in Google Drive. Set the two paths below. The project archive should preserve its folder structure (so the project root contains `app.py`, `backend/`, `models/`, `model.ckpt`, `dsen2-cr-main/`, and `diffffusiosm/`).

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
from pathlib import Path
import zipfile, shutil, os, sys

PROJECT_ZIP = Path('/content/drive/MyDrive/colab_project_bundle.zip')  # edit if stored elsewhere
DATA_ZIP = Path('/content/drive/MyDrive/sen12ms_cr_50_pairs.zip')          # edit if stored elsewhere
PROJECT_DIR = Path('/content/project')
DATA_DIR = Path('/content/scene_pairs')
for archive, destination in ((PROJECT_ZIP, PROJECT_DIR), (DATA_ZIP, DATA_DIR)):
    if not archive.is_file():
        raise FileNotFoundError(f'Upload or correct this Drive path: {archive}')
    destination.mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(archive) as zf:
        zf.extractall(destination)

# Find the actual project root, even when the zip has one enclosing directory.
project_candidates = [p.parent for p in PROJECT_DIR.rglob('app.py') if (p.parent / 'backend').is_dir() and (p.parent / 'models').is_dir()]
if not project_candidates:
    raise FileNotFoundError('Could not find project root containing app.py, backend/, and models/.')
PROJECT_ROOT = project_candidates[0]
sys.path.insert(0, str(PROJECT_ROOT))
print('Project:', PROJECT_ROOT)
print('Dataset:', DATA_DIR)

## 3. Configure and validate the scene pairs
Expected layout per folder: `s1.tif`, `s2_cloudy.tif`, `ground_truth.tif`, optional `valid_mask.tif`. Adjust names here if yours differ.

In [ ]:
import numpy as np
import rasterio
from rasterio.warp import reproject, Resampling
from rasterio.enums import Resampling as RioResampling
from pathlib import Path

SCENE_PAIRS = []
for folder in sorted(p for p in DATA_DIR.rglob('*') if p.is_dir()):
    s1 = folder / 's1.tif'
    cloudy = folder / 's2_cloudy.tif'
    gt = folder / 'ground_truth.tif'
    mask = folder / 'valid_mask.tif'
    if s1.is_file() and cloudy.is_file() and gt.is_file():
        SCENE_PAIRS.append({'name': folder.name, 's1': s1, 'cloudy': cloudy, 'gt': gt,
                            'mask': mask if mask.is_file() else None})
if not SCENE_PAIRS:
    raise ValueError(f'Found {len(SCENE_PAIRS)} complete scene folders; expected at least one. Check zip layout/file names.')

def validate_scene(scene):
    with rasterio.open(scene['s1']) as a, rasterio.open(scene['cloudy']) as b, rasterio.open(scene['gt']) as c:
        if a.count != 2: raise ValueError(f"{scene['name']}: S1 must have VV,VH (2 bands), got {a.count}")
        if b.count != 13 or c.count != 13: raise ValueError(f"{scene['name']}: cloudy and GT must each have 13 bands; got {b.count}, {c.count}")
        for label, src in [('S1', a), ('GT', c)]:
            if (src.width, src.height, src.crs) != (b.width, b.height, b.crs) or not src.transform.almost_equals(b.transform):
                raise ValueError(f"{scene['name']}: {label} is not co-registered with cloudy S2. Align all rasters first.")
        if scene['mask']:
            with rasterio.open(scene['mask']) as m:
                if m.count != 1 or (m.width, m.height, m.crs) != (b.width, b.height, b.crs) or not m.transform.almost_equals(b.transform):
                    raise ValueError(f"{scene['name']}: validity mask must be one band and match the S2 grid")
        return (b.width, b.height), str(b.crs), b.transform

for item in SCENE_PAIRS:
    item['grid'], item['crs'], item['transform'] = validate_scene(item)
print(f'Validated {len(SCENE_PAIRS)} co-registered scenes.')
print([(x['name'], x['grid'], x['crs']) for x in SCENE_PAIRS])

## 4. Load the project models on the T4
Former uses `model.ckpt`. CNN uses the converted PyTorch weights `model_SARcarl.torch.npz`; this avoids needing the Windows-only Python 3.7 conversion environment in Colab.

In [ ]:
from backend.services.processing.inference_service import InferenceService, InferenceConfig
from backend.services.processing.dsen2cr_service import DSen2CRService
from pathlib import Path

FORMER_CKPT = PROJECT_ROOT / 'model.ckpt'
DSEN_NPZ = PROJECT_ROOT / 'dsen2-cr-main' / 'checkpoints' / 'model_SARcarl.torch.npz'
if not FORMER_CKPT.is_file(): raise FileNotFoundError(f'Missing {FORMER_CKPT}')
if not DSEN_NPZ.is_file(): raise FileNotFoundError(f'Missing converted DSen2-CR weights: {DSEN_NPZ}. Run CNN once in the desktop app to generate this file, then include it in the bundle.')

class ColabDSen2CRService(DSen2CRService):
    @property
    def available(self):
        return self.torch_weights.is_file()
    def _ensure_torch_weights(self):
        if not self.torch_weights.is_file():
            raise FileNotFoundError(f'Missing converted weights: {self.torch_weights}')

former_service = InferenceService(checkpoint_path=FORMER_CKPT, device='cuda')
cnn_service = ColabDSen2CRService(project_root=PROJECT_ROOT)
print('Former checkpoint:', FORMER_CKPT.stat().st_size / 1e6, 'MB')
print('CNN converted weights:', DSEN_NPZ.stat().st_size / 1e6, 'MB')
print('Both models configured for:', torch.cuda.get_device_name(0))

## 5. Run cloud removal and evaluate against ground truth
Metrics are calculated only on valid, finite pixels. Model outputs are evaluated in reflectance [0,1] across all 13 bands. Results and predictions are saved under `/content/evaluation_outputs/`.

In [ ]:
import pandas as pd
from skimage.metrics import structural_similarity
from backend.services.processing.evaluation_metrics import OPENSR_BAND_NAMES

OUT_DIR = Path('/content/evaluation_outputs')
OUT_DIR.mkdir(exist_ok=True)
BAND_NAMES = ['B01','B02','B03','B04','B05','B06','B07','B08','B8A','B09','B10','B11','B12']
def load_gt_and_mask(scene):
    with rasterio.open(scene['gt']) as src:
        gt = src.read().astype(np.float32)
        profile = src.profile.copy()
        nodata = src.nodata
    # GT may be stored as Sentinel DN (0..10000) or reflectance (0..1).
    finite = np.isfinite(gt).all(axis=0)
    if nodata is not None: finite &= np.all(gt != nodata, axis=0)
    if scene['mask']:
        with rasterio.open(scene['mask']) as src: finite &= src.read(1) != 0
    gt = np.clip(gt / 10000.0 if np.nanmax(gt) > 1.5 else gt, 0.0, 1.0)
    return gt, finite, profile

def calc_metrics(pred, target, valid):
    pred = np.clip(np.nan_to_num(pred), 0, 1)
    target = np.clip(np.nan_to_num(target), 0, 1)
    valid = valid & np.isfinite(pred).all(axis=0) & np.isfinite(target).all(axis=0)
    if valid.sum() == 0: raise ValueError('No finite valid pixels for metric calculation.')
    diff = pred[:, valid] - target[:, valid]
    mae = float(np.abs(diff).mean()); rmse = float(np.sqrt(np.square(diff).mean()))
    psnr = float('inf') if rmse == 0 else float(20 * np.log10(1.0 / rmse))
    dot = np.sum(pred * target, axis=0); norms = np.linalg.norm(pred, axis=0) * np.linalg.norm(target, axis=0)
    spectral_valid = valid & (norms > 1e-8)
    sam = float(np.degrees(np.arccos(np.clip(dot[spectral_valid] / norms[spectral_valid], -1, 1))).mean()) if spectral_valid.any() else float('nan')
    # Average the SSIM maps only over valid pixels rather than counting nodata/background.
    ssim_values = []
    for band in range(pred.shape[0]):
        _, ssim_map = structural_similarity(target[band], pred[band], data_range=1.0, full=True)
        ssim_values.append(float(np.mean(ssim_map[valid])))
    return {'MAE': mae, 'RMSE': rmse, 'PSNR_dB': psnr, 'SSIM': float(np.mean(ssim_values)), 'SAM_deg': sam, 'Pixels': int(valid.sum())}

def write_prediction(path, pred, profile):
    out_profile = profile.copy(); out_profile.update(count=13, dtype='float32', nodata=0.0, compress='deflate')
    with rasterio.open(path, 'w', **out_profile) as dst:
        dst.write(np.asarray(pred, dtype=np.float32))
        for i, name in enumerate(BAND_NAMES, 1): dst.set_band_description(i, name)

rows = []
for n, scene in enumerate(SCENE_PAIRS, 1):
    print("[{}/{}] {}".format(n, len(SCENE_PAIRS), scene["name"]))
    gt, valid, profile = load_gt_and_mask(scene)
    mask_path = scene['mask']
    former = former_service.run(scene['s1'], scene['cloudy'], mask_path,
        config=InferenceConfig(device='cuda', tile_size=128, overlap=32))
    cnn = cnn_service.run(scene['s1'], scene['cloudy'], mask_path, brightness=2.5)
    for model_name, result in [('Former', former), ('CNN', cnn)]:
        metrics = calc_metrics(result.prediction, gt, valid)
        out_path = OUT_DIR / f"{scene['name']}_{model_name.lower()}_cloud_free_13band.tif"
        write_prediction(out_path, result.prediction, profile)
        rows.append({'Scene': scene['name'], 'Model': model_name, **metrics})

metrics_df = pd.DataFrame(rows)
display(metrics_df)
summary_df = metrics_df.groupby('Model')[['MAE','RMSE','PSNR_dB','SSIM','SAM_deg','Pixels']].agg(['mean','std'])
display(summary_df)
metrics_df.to_csv(OUT_DIR / 'per_scene_cloud_removal_metrics.csv', index=False)
summary_df.to_csv(OUT_DIR / 'cloud_removal_mean_std.csv')
print('Saved per-scene and aggregate cloud-removal metrics.')

## 6. Optional: OpenSR on both cloud-removal outputs and score against ground truth
This runs OpenSR 20 times (10 scenes × 2 models) and can take a while. OpenSR output is 4-band B04/B03/B02/B08 at 2.5 m. For a 10 m ground truth, the SR raster is area-averaged to the GT grid before comparison. These scores assess 10 m agreement only, not native 2.5 m detail.

In [ ]:
RUN_OPENSR = True  # Set False to skip the long super-resolution stage
if RUN_OPENSR:
    !pip -q install opensr-utils==1.1.1 diffusers transformers pytorch-lightning omegaconf kornia lpips
    import sys
    from backend.services.processing.super_resolution.opensr_service import OpenSRService, OpenSRConfig
    opensr_dir = PROJECT_ROOT / 'diffffusiosm'
    if not (opensr_dir / 'process_tif.py').is_file(): raise FileNotFoundError(f'Missing OpenSR project: {opensr_dir}')
    sr_service = OpenSRService(OpenSRConfig(teammate_dir=opensr_dir, python_executable=Path(sys.executable),
        script_path=opensr_dir / 'process_tif.py', output_dir=OUT_DIR / 'opensr',
        model_version='v2-preview', timeout_seconds=1800))
    sr_rows = []
    # OpenSR returns four bands in B04, B03, B02, B08 order.
    gt_indexes = [3, 2, 1, 7]
    for scene in SCENE_PAIRS:
        gt13, valid, gt_profile = load_gt_and_mask(scene)
        target4 = gt13[gt_indexes]
        for model_name in ('former', 'cnn'):
            source = OUT_DIR / f"{scene['name']}_{model_name}_cloud_free_13band.tif"
            sr_result = sr_service.run(source, output_dir=OUT_DIR / 'opensr')
            sr_path = Path(sr_result.sr_geotiff_path)
            with rasterio.open(sr_path) as src:
                sr = src.read().astype(np.float32)
                sr_transform, sr_crs = src.transform, src.crs
                sr_nodata = src.nodata
            if sr.shape[0] != 4: raise ValueError(f'Expected 4-band OpenSR result, got {sr.shape}')
            if np.nanmax(sr) > 1.5: sr = sr / 10000.0
            # SEN12MS-CR TIFFs in this archive have no CRS metadata. OpenSR is an exact 4x
            # scale, so average each 4x4 pixel block back to the paired 256x256 reference grid.
            ref_h, ref_w = gt_profile['height'], gt_profile['width']
            if sr.shape[1:] != (ref_h * 4, ref_w * 4):
                raise ValueError(f'Expected exact 4x OpenSR dimensions {(ref_h*4, ref_w*4)}, got {sr.shape[1:]}')
            aligned = sr.reshape(4, ref_h, 4, ref_w, 4).mean(axis=(2, 4))
            sr_valid = valid & np.isfinite(aligned).all(axis=0)
            metrics = calc_metrics(aligned, target4, sr_valid)
            sr_rows.append({'Scene': scene['name'], 'Model': model_name.title() + ' + OpenSR', **metrics})
    sr_metrics_df = pd.DataFrame(sr_rows)
    display(sr_metrics_df)
    sr_summary_df = sr_metrics_df.groupby('Model')[['MAE','RMSE','PSNR_dB','SSIM','SAM_deg','Pixels']].agg(['mean','std'])
    display(sr_summary_df)
    sr_metrics_df.to_csv(OUT_DIR / 'per_scene_opensr_metrics.csv', index=False)
    sr_summary_df.to_csv(OUT_DIR / 'opensr_mean_std.csv')
    print('Saved OpenSR metrics. SR scores use the four matched bands at the reference grid.')

## 7. Download the tables and prediction archive
The main results are in `/content/evaluation_outputs`. Zip all per-scene GeoTIFF predictions and CSVs, then download the archive.

In [ ]:
import shutil
archive_path = shutil.make_archive('/content/cloud_removal_research_results', 'zip', OUT_DIR)
from google.colab import files
files.download(archive_path)
display(metrics_df.groupby('Model')[['MAE','RMSE','PSNR_dB','SSIM','SAM_deg']].agg(['mean','std']))
if RUN_OPENSR: display(sr_metrics_df.groupby('Model')[['MAE','RMSE','PSNR_dB','SSIM','SAM_deg']].agg(['mean','std']))

## Research reporting notes
- Report the **number of independent scenes (n=50)** and provide mean ± standard deviation; include per-scene results in supplementary material.
- Describe the ground-truth acquisition dates, temporal gaps, co-registration/resampling, radiometric scaling, mask policy, band order, and exact checkpoint versions.
- Smaller MAE/RMSE/SAM are better; larger PSNR/SSIM are better. These are fidelity metrics, not a single “accuracy” percentage.
- The displayed OpenSR evaluation is four-band 10 m-equivalent agreement when the target is 10 m. Do not describe it as measured 2.5 m reconstruction accuracy without 2.5 m reference imagery.
- Ten scenes are a small evaluation sample. Preserve per-scene results and avoid claiming broad generalization from the mean alone.